## <center style="color:blue;">**FootVerse**</center>

### <center>**Modélisation et Analyse de Données Footballistiques**</center>

Ce projet a pour objectif de collecter des données de football à l’aide du web scraping (Selenium), puis de les transformer et nettoyer afin d’assurer leur qualité. Les données seront ensuite modélisées et stockées dans une base de données PostgreSQL. Enfin, un modèle de machine learning sera entraîné pour prédire l’équipe gagnante d’un match.

In [92]:
import os
import shutil

src = "../data/bronze/"
dest = "../data/silver/"

shutil.copy(os.path.join(src,"saisons.csv"), os.path.join(dest,"saisons.csv"))

shutil.copy(os.path.join(src,"teams.csv"), os.path.join(dest,"teams.csv"))

print("Fichiers Copiés avec Succés !")

Fichiers Copiés avec Succés !


<br>

### <span style="color:green;">**Gérer les Tables des Joueurs :**</span>

#### <span style="color:orange;">**1. Gérer les Valeurs Manquantes :**</span>

In [93]:
import pandas as pd

bronze_path = "../data/bronze/teams"

equipes = []

# Récupérer les noms des Equipes
for team in os.listdir(bronze_path) :
    equipes.append(team)

for equipe in equipes :
    equipe_path = os.path.join(bronze_path, equipe, "players.csv")

    # Lire le fichier csv des joueurs comme DataFrame
    players_df = pd.read_csv(equipe_path)

    # Identifier le nombre des valeurs manquantes
    val_nulls = players_df.isnull().sum()
    print(f"- Nombre de Valeurs Manquantes dans l'Equipe '{equipe}' est : \n{val_nulls}")

    # Remplacer les lignes (axis=0) contenant les valeurs manquantes avec 0
    players_df = players_df.fillna(0)

    silver_path = "../data/silver"

    os.makedirs(os.path.join(silver_path, "teams", equipe), exist_ok=True)

    dest_path = f"../data/silver/teams/{equipe}"

    # Enregistrer DataFrame sous format csv
    players_df.to_csv(os.path.join(dest_path, "players.csv"), index=False)


- Nombre de Valeurs Manquantes dans l'Equipe 'Arsenal' est : 
Player     0
Nation     0
Pos        0
Age        0
MP         0
Starts     0
Min       13
90s       13
Gls       13
Ast       13
G+A       13
G-PK      13
PK        13
PKatt     13
CrdY      13
CrdR      13
dtype: int64
- Nombre de Valeurs Manquantes dans l'Equipe 'Aston Villa' est : 
Player    0
Nation    0
Pos       0
Age       0
MP        0
Starts    0
Min       6
90s       6
Gls       6
Ast       6
G+A       6
G-PK      6
PK        6
PKatt     6
CrdY      6
CrdR      6
dtype: int64
- Nombre de Valeurs Manquantes dans l'Equipe 'Bournemouth' est : 
Player    0
Nation    0
Pos       0
Age       0
MP        0
Starts    0
Min       9
90s       9
Gls       9
Ast       9
G+A       9
G-PK      9
PK        9
PKatt     9
CrdY      9
CrdR      9
dtype: int64
- Nombre de Valeurs Manquantes dans l'Equipe 'Brentford' est : 
Player    0
Nation    0
Pos       0
Age       0
MP        0
Starts    0
Min       7
90s       7
Gls       7
Ast

#### <span style="color:orange;">**2. Gérer les Doublons :**</span>

In [94]:
silver_path = "../data/silver/teams"

nulls = []

for equipe in equipes :
    equipe_path = os.path.join(silver_path, equipe, "players.csv")

    # Lire le fichier csv des joueurs comme DataFrame
    players_df = pd.read_csv(equipe_path)

    # Identifier le nombre des valeurs manquantes
    val_nulls = players_df.duplicated().sum()
    

    nulls.append(val_nulls)

if sum(nulls) == 0 :
    print(f"Toutes les Equipes ne contiennent pas des Doublons !")

Toutes les Equipes ne contiennent pas des Doublons !


#### <span style="color:orange;">**3. Standarisation des Données :**</span>

##### **3.1. Copier les Fichiers CSV des Joueurs de Chaque Equipe en ajoutant la colonne ``SQUAD`` :**

In [95]:
for equipe in equipes :
    players_df = pd.read_csv(os.path.join(silver_path, equipe, "players.csv"))

    cols = list(players_df.columns)

    players_df["Squad"] = equipe

    players_df.to_csv(os.path.join(silver_path, equipe, "players.csv"), index=False)


##### **3.2. Créer un fichier CSV combinant tous les Joueurs des Equipes :**

In [96]:
dfs = []

for equipe in equipes:
    path = os.path.join(silver_path ,equipe ,"players.csv")
    players_df = pd.read_csv(path)
    dfs.append(players_df)        

df = pd.concat(dfs, ignore_index=True)

df.to_csv("../data/silver/players.csv", index=False)

##### **3.3. Importer les Fichiers des Joueurs :**

In [97]:
import pandas as pd

df = pd.read_csv("../data/silver/players.csv")

df.head()

,Player,Nation,Pos,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Squad
0,David Raya,es ESP,GK,28.0,38,38,"3,420",38.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,Arsenal
1,William Saliba,fr FRA,DF,23.0,35,35,"3,039",33.8,2.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,Arsenal
2,Declan Rice,eng ENG,MF,25.0,35,33,"2,825",31.4,4.0,7.0,11.0,4.0,0.0,0.0,7.0,1.0,Arsenal
3,Thomas Partey,gh GHA,"MF,DF",31.0,35,31,"2,797",31.1,4.0,2.0,6.0,4.0,0.0,0.0,4.0,0.0,Arsenal
4,Leandro Trossard,be BEL,FW,29.0,38,28,"2,546",28.3,8.0,7.0,15.0,8.0,0.0,0.0,4.0,1.0,Arsenal


##### **3.4. Standariser la Colonne ``Nation`` :**

In [98]:
df["Nation"] = df["Nation"].apply(
    lambda row : "Inconnu" if row == "0" else row.split(' ')[1]
)

df.head()

,Player,Nation,Pos,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Squad
0,David Raya,ESP,GK,28.0,38,38,"3,420",38.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,Arsenal
1,William Saliba,FRA,DF,23.0,35,35,"3,039",33.8,2.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,Arsenal
2,Declan Rice,ENG,MF,25.0,35,33,"2,825",31.4,4.0,7.0,11.0,4.0,0.0,0.0,7.0,1.0,Arsenal
3,Thomas Partey,GHA,"MF,DF",31.0,35,31,"2,797",31.1,4.0,2.0,6.0,4.0,0.0,0.0,4.0,0.0,Arsenal
4,Leandro Trossard,BEL,FW,29.0,38,28,"2,546",28.3,8.0,7.0,15.0,8.0,0.0,0.0,4.0,1.0,Arsenal


##### **3.5. Standariser la Colonne ``Min`` :**

In [99]:
df["Min"] = df["Min"].apply(
    lambda row : int(str(row).replace(",", ""))
)

df.head()

,Player,Nation,Pos,Age,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,Squad
0,David Raya,ESP,GK,28.0,38,38,3420,38.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,Arsenal
1,William Saliba,FRA,DF,23.0,35,35,3039,33.8,2.0,0.0,2.0,2.0,0.0,0.0,2.0,1.0,Arsenal
2,Declan Rice,ENG,MF,25.0,35,33,2825,31.4,4.0,7.0,11.0,4.0,0.0,0.0,7.0,1.0,Arsenal
3,Thomas Partey,GHA,"MF,DF",31.0,35,31,2797,31.1,4.0,2.0,6.0,4.0,0.0,0.0,4.0,0.0,Arsenal
4,Leandro Trossard,BEL,FW,29.0,38,28,2546,28.3,8.0,7.0,15.0,8.0,0.0,0.0,4.0,1.0,Arsenal


##### **3.6. Enregistrer la DataFrame après Standarisation :**

In [100]:
df.to_csv("../data/silver/players.csv", index=False)

<br>

### <span style="color:green;">**Gérer les Tables des Statistiques des Matchs :**</span>

#### <span style="color:orange;">**1. Gérer les Valeurs Manquantes :**</span>

##### **1.1. Récupérer les Noms des Equipes :**

In [101]:
import pandas as pd

silver_path = "../data/silver/teams"

equipes = []

# Récupérer les noms des Equipes
for team in os.listdir(silver_path) :
    equipes.append(team)

##### **1.2. Identifier les Colonnes qui Contiennent des Valeurs Manquantes :**

In [102]:
import pandas as pd

bronze_path = "../data/bronze/teams"

null_colonnes = []

for equipe in equipes :
    equipe_path = os.path.join(bronze_path, equipe, "scores.csv")

    # Lire le fichier csv des joueurs comme DataFrame
    scores_df = pd.read_csv(equipe_path)

    # Identifier tous les colonnes contenant des valeurs manquantes
    colonnes = list(scores_df.columns[scores_df.isnull().any()])

    # Stocker les colonnes 
    for col in colonnes :
        null_colonnes.append(col)

# Eliminer les colonnes doublons
null_cols = list(set(null_colonnes))

null_cols
    

['xG', 'xGA', 'Attendance', 'Poss', 'Referee']

##### **1.3. Remplacer les Valeurs Manquantes de chaque Colonne par la valeur convenable :**

In [103]:
import numpy as np

silver_path = "../data/silver/teams"

for equipe in equipes :
    equipe_path = os.path.join(bronze_path, equipe, "scores.csv")

    scores_df = pd.read_csv(equipe_path)

    cols = list(scores_df.columns[scores_df.isnull().any()])

    for col in cols :
        if col in ['xG','xGA'] :
            scores_df[col] = scores_df[col].fillna(0)
        elif col == 'Poss':
            mean_col = f"{scores_df[col].mean():.2f}"
            scores_df[col] = scores_df[col].fillna(mean_col)
        elif col == 'Referee':
            scores_df[col] = scores_df[col].fillna("Inconnu")
        elif col == 'Attendance' :
            scores_df[col] = scores_df[col].apply(lambda x : int(x.replace(",", "")) if isinstance(x, str) else np.nan )
            mean_col = f"{scores_df[col].mean():.2f}"
            scores_df[col] = scores_df[col].fillna(mean_col)
    
    scores_df.to_csv(os.path.join(silver_path,equipe,"scores.csv"), index=False)

    print(f"- {equipe} : Done !")

print("Valeurs Manquées Remplacées avec Succés !")


- Arsenal : Done !
- Aston Villa : Done !
- Bournemouth : Done !
- Brentford : Done !
- Brighton : Done !
- Chelsea : Done !
- Crystal Palace : Done !
- Everton : Done !
- Fulham : Done !
- Ipswich Town : Done !
- Leicester City : Done !
- Liverpool : Done !
- Manchester City : Done !
- Manchester Utd : Done !
- Newcastle Utd : Done !
- Nott'ham Forest : Done !
- Southampton : Done !
- Tottenham : Done !
- West Ham : Done !
- Wolves : Done !
Valeurs Manquées Remplacées avec Succés !


#### <span style="color:orange;">**2. Gérer les Doublons :**</span>

##### **2.1. Copier les Fichiers CSV des Statistiques des Matchs de Chaque Equipe en ajoutant la colonne ``SQUAD`` :**

In [104]:
for equipe in equipes :
    scores_df = pd.read_csv(os.path.join(silver_path, equipe, "scores.csv"))

    cols = list(scores_df.columns)

    scores_df["Squad"] = equipe

    new_cols = [cols[0], "Squad"] + cols[1::]

    scores_df = scores_df[new_cols]

    scores_df.to_csv(os.path.join(silver_path, equipe, "scores.csv"), index=False)


##### **2.2. Créer un fichier CSV combinant tous les Matchs des Equipes :**

In [105]:
dfs = []

for equipe in equipes:
    path = os.path.join(silver_path, equipe, "scores.csv")
    scores_df = pd.read_csv(path)
    dfs.append(scores_df)        

df = pd.concat(dfs, ignore_index=True)

df.to_csv("../data/silver/scores.csv", index=False)

##### **2.3. Supprimer les Doublons dans le fichier CSV global des Statistiques de tous les Matches :**

In [106]:
df = pd.read_csv("../data/silver/scores.csv")

df["Resume"] = df.apply(
    lambda x : f"{x['Date']}_{x['Time']}_"+"_".join(sorted([x["Squad"], x["Opponent"]])),
    axis=1
)

df = df.drop_duplicates(subset="Resume")

df.to_csv("../data/silver/scores.csv", index=False)


<br>

#### <span style="color:orange;">**3. Standarisation des Données :**</span>

##### **3.1. importer les Fichiers des Statistiques des Matchs :**

In [107]:
import pandas as pd

df = pd.read_csv("../data/silver/scores.csv")

df.head()

,Date,Squad,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,xG,xGA,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Resume
0,2024-08-17,Arsenal,15:00,Premier League,Matchweek 1,Sat,Home,W,2,0,Wolves,1.2,0.5,53.0,60261.0,Martin Ødegaard,4-3-3,4-2-3-1,Jarred Gillett,2024-08-17_15:00_Arsenal_Wolves
1,2024-08-24,Arsenal,17:30,Premier League,Matchweek 2,Sat,Away,W,2,0,Aston Villa,0.9,1.2,60.0,41587.0,Martin Ødegaard,4-3-3,4-4-2,Michael Oliver,2024-08-24_17:30_Arsenal_Aston Villa
2,2024-08-31,Arsenal,12:30,Premier League,Matchweek 3,Sat,Home,D,1,1,Brighton,2.1,1.7,36.0,60326.0,Martin Ødegaard,4-3-3,4-2-3-1,Chris Kavanagh,2024-08-31_12:30_Arsenal_Brighton
3,2024-09-15,Arsenal,14:00,Premier League,Matchweek 4,Sun,Away,W,1,0,Tottenham,0.7,0.7,37.0,61645.0,Jorginho,4-4-2,4-3-3,Jarred Gillett,2024-09-15_14:00_Arsenal_Tottenham
4,2024-09-19,Arsenal,21:00 (20:00),Champions Lg,League phase,Thu,Away,D,0,0,it Atalanta,0.8,1.2,46.0,22858.0,Gabriel Jesus,4-3-3,3-4-3,Clément Turpin,2024-09-19_21:00 (20:00)_Arsenal_it Atalanta


##### **3.2. Standariser la Colonne ``Time`` :**

In [108]:
df["Time"] = df["Time"].apply(
    lambda row : row.split('(')[0]
)

df["Time"].head()

df

,Date,Squad,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,xG,xGA,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Resume
0,2024-08-17,Arsenal,15:00,Premier League,Matchweek 1,Sat,Home,W,2,0,Wolves,1.2,0.5,53.00,60261.0,Martin Ødegaard,4-3-3,4-2-3-1,Jarred Gillett,2024-08-17_15:00_Arsenal_Wolves
1,2024-08-24,Arsenal,17:30,Premier League,Matchweek 2,Sat,Away,W,2,0,Aston Villa,0.9,1.2,60.00,41587.0,Martin Ødegaard,4-3-3,4-4-2,Michael Oliver,2024-08-24_17:30_Arsenal_Aston Villa
2,2024-08-31,Arsenal,12:30,Premier League,Matchweek 3,Sat,Home,D,1,1,Brighton,2.1,1.7,36.00,60326.0,Martin Ødegaard,4-3-3,4-2-3-1,Chris Kavanagh,2024-08-31_12:30_Arsenal_Brighton
3,2024-09-15,Arsenal,14:00,Premier League,Matchweek 4,Sun,Away,W,1,0,Tottenham,0.7,0.7,37.00,61645.0,Jorginho,4-4-2,4-3-3,Jarred Gillett,2024-09-15_14:00_Arsenal_Tottenham
4,2024-09-19,Arsenal,21:00,Champions Lg,League phase,Thu,Away,D,0,0,it Atalanta,0.8,1.2,46.00,22858.0,Gabriel Jesus,4-3-3,3-4-3,Clément Turpin,2024-09-19_21:00 (20:00)_Arsenal_it Atalanta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,2024-12-09,West Ham,20:00,Premier League,Matchweek 15,Mon,Home,W,2,1,Wolves,1.0,1.4,54.00,"62,474",Jarrod Bowen,4-2-3-1,3-4-3,John Brooks,2024-12-09_20:00 (21:00)_West Ham_Wolves
555,2025-04-01,West Ham,19:45,Premier League,Matchweek 30,Tue,Away,L,0,1,Wolves,1.3,1.2,58.00,"29,587",Jarrod Bowen,3-4-3,3-4-3,Tony Harrington,2025-04-01_19:45 (18:45)_West Ham_Wolves
556,2024-08-28,Wolves,19:30,EFL Cup,Second round,Wed,Home,W,2,0,Burnley,0.0,0.0,50.00,"19,236",Craig Dawson,4-2-3-1,4-4-1-1,Joshua Smith,2024-08-28_19:30_Burnley_Wolves
557,2025-01-11,Wolves,12:00,FA Cup,Third round proper,Sat,Away,W,2,1,Bristol City,0.0,0.0,47.86,"23,485",Matt Doherty,3-4-3,3-4-3,Robert Jones,2025-01-11_12:00 (13:00)_Bristol City_Wolves


##### **3.3. Standariser la Colonne ``Formation`` : :**

In [109]:
df.iloc[462:465, 16:17]

,Formation
462,4-1-2-1-2◆
463,4-2-3-1
464,4-1-4-1


In [110]:
df["Formation"] = df["Formation"].apply(
    lambda row : row.split('◆')[0]
)

df.iloc[462:465, 16:17]

,Formation
462,4-1-2-1-2
463,4-2-3-1
464,4-1-4-1


##### **3.4. Standariser la Colonne ``Attendance` :**

In [111]:
df["Attendance"] = df["Attendance"].apply(
    lambda row : float(str(row).replace(",", ""))
)

df.tail()

,Date,Squad,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,xG,xGA,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Resume
554,2024-12-09,West Ham,20:00,Premier League,Matchweek 15,Mon,Home,W,2,1,Wolves,1.0,1.4,54.00,62474.0,Jarrod Bowen,4-2-3-1,3-4-3,John Brooks,2024-12-09_20:00 (21:00)_West Ham_Wolves
555,2025-04-01,West Ham,19:45,Premier League,Matchweek 30,Tue,Away,L,0,1,Wolves,1.3,1.2,58.00,29587.0,Jarrod Bowen,3-4-3,3-4-3,Tony Harrington,2025-04-01_19:45 (18:45)_West Ham_Wolves
556,2024-08-28,Wolves,19:30,EFL Cup,Second round,Wed,Home,W,2,0,Burnley,0.0,0.0,50.00,19236.0,Craig Dawson,4-2-3-1,4-4-1-1,Joshua Smith,2024-08-28_19:30_Burnley_Wolves
557,2025-01-11,Wolves,12:00,FA Cup,Third round proper,Sat,Away,W,2,1,Bristol City,0.0,0.0,47.86,23485.0,Matt Doherty,3-4-3,3-4-3,Robert Jones,2025-01-11_12:00 (13:00)_Bristol City_Wolves
558,2025-02-09,Wolves,12:30,FA Cup,Fourth round proper,Sun,Away,W,2,0,Blackburn,0.0,0.0,53.00,15141.0,Nélson Semedo,3-4-3,4-2-3-1,Lewis Smith,2025-02-09_12:30 (13:30)_Blackburn_Wolves


##### **3.5. Enregistrer la DataFrame après Standarisation :**

In [112]:
df.to_csv("../data/silver/scores.csv", index=False)